In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd
#Got the following code from dataset website: https://huggingface.co/datasets/osv5m/osv5m
#This downloads the specific zips from test.zip images  (00.zip - 04.zip) as well as the readme instructions
for i in range(5):
    hf_hub_download(repo_id="osv5m/osv5m", filename=str(i).zfill(2)+'.zip', subfolder="images/test", repo_type='dataset', local_dir="datasets/OpenWorld")
    hf_hub_download(repo_id="osv5m/osv5m", filename="README.md", repo_type='dataset', local_dir="datasets/OpenWorld")

images/test/00.zip: reconstructing file:   0%|          |  0.00B / 2.25GB            

images/test/00.zip: downloading bytes:           |  0.00B            

README.md:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

images/test/01.zip: reconstructing file:   0%|          |  0.00B / 2.24GB            

images/test/01.zip: downloading bytes:           |  0.00B            

...

images/test/02.zip: downloading bytes:           |  0.00B            

images/test/03.zip: reconstructing file:   0%|          |  0.00B / 2.25GB            

images/test/03.zip: downloading bytes:           |  0.00B            

images/test/04.zip: reconstructing file:   0%|          |  0.00B /  453MB            

images/test/04.zip: downloading bytes:           |  0.00B            

In [12]:

#This downloads the test.csv file
csv_path = hf_hub_download(
    repo_id="osv5m/osv5m",
    filename="test.csv",
    repo_type="dataset",
    local_dir="datasets/OpenWorld"
)

#Verifies the shape
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())


Dataset shape: (210122, 26)
                 id   latitude  longitude  \
0   547473234108938 -16.336027  45.628280   
1   826109781317024  50.855687  56.147997   
2  1006398440000844  37.956651  14.954485   
3  2943891539215481  12.373333  -8.909906   
4   122945119799579   7.510295  99.061884   

                                  thumb_original_url country  \
0  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      MG   
1  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      KZ   
2  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      IT   
3  https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...      ML   
4  https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...      TH   

                 sequence    captured_at  lon_bin  lat_bin      cell  ...  \
0  Z2sJMyDVAtHBu7UFGvnYXg  1660909818797       61       28  (61, 28)  ...   
1  6cTpNBSYQH6SeDsfz_0mEQ  1548919530404       64       78  (64, 78)  ...   
2  xhcEpYg3dKnzts5ejIiSBJ  1658912403000       53       68  (53, 68)  ...   
3  j6nwj

In [ ]:

from pathlib import Path
import zipfile

#This extracts the image data
#WARNING WARNING WARNING
#This data is huge already in zip files, EXPECT 20GB to 30GB at least used
base_dir = Path("datasets/OpenWorld")
image_dir = base_dir / "extracted_test"
image_dir.mkdir(parents=True, exist_ok=True)

for i in range(5):
    zip_path = base_dir / "images" / "test" / f"{i:02d}.zip"

    if not zip_path.exists():
        zip_path = base_dir / f"{i:02d}.zip"

    if zip_path.exists():
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(image_dir)
        print(f"Extracted {zip_path.name}")
    else:
        print(f"Archive not found: {zip_path}")


Extracted 00.zip
Extracted 01.zip
Extracted 02.zip
Extracted 03.zip
Extracted 04.zip


In [30]:

#This confirms data loaded as expected
print("Metadata rows:", len(df))
print("Metadata columns:", len(df.columns))

#nunique is num of unique elements
print("Unique countries:", df["country"].nunique())

print("Country counts:")
print(df["country"].value_counts().head())

image_files = [
    p for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {
        ".jpg"
    }
]

print("Extracted image files:", len(image_files))
print("Example image paths:", image_files[:5])


Metadata rows: 210122
Metadata columns: 27
Unique countries: 219

Country counts:
country
US    23875
RU    14984
AU    11052
BR    10425
CA     9580
Name: count, dtype: int64
Extracted image files: 210122
Example image paths: [WindowsPath('datasets/OpenWorld/extracted_test/00/1000055667540054.jpg'), WindowsPath('datasets/OpenWorld/extracted_test/00/1000272590507984.jpg'), WindowsPath('datasets/OpenWorld/extracted_test/00/1000299547168698.jpg'), WindowsPath('datasets/OpenWorld/extracted_test/00/1000344074062646.jpg'), WindowsPath('datasets/OpenWorld/extracted_test/00/1000506854093129.jpg')]


In [ ]:
from pathlib import Path


image_dir = Path("datasets/OpenWorld/extracted_test")

# Find images inside extracted_test and all its subfolders
image_lookup = {
    p.stem: str(p)
    for p in image_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg"}
}

# Match image IDs to the CSV
df["id"] = df["id"].astype(str).str.strip()
df["image_path"] = df["id"].map(image_lookup)

print("Unique image IDs:", len(image_lookup))
print("Unique CSV IDs:", df["id"].nunique())
print("Matching IDs:", df["image_path"].notna().sum())
print("CSV IDs missing images:", df["image_path"].isna().sum())

print("\nExample matches:")
print(df[["id", "image_path", "country"]].head())

Unique image IDs: 210122
Unique CSV IDs: 210122
Matching IDs: 210122
CSV IDs missing images: 0

Example matches:
                 id                                         image_path country
0   547473234108938  datasets\OpenWorld\extracted_test\03\547473234...      MG
1   826109781317024  datasets\OpenWorld\extracted_test\01\826109781...      KZ
2  1006398440000844  datasets\OpenWorld\extracted_test\00\100639844...      IT
3  2943891539215481  datasets\OpenWorld\extracted_test\00\294389153...      ML
4   122945119799579  datasets\OpenWorld\extracted_test\00\122945119...      TH


In [ ]:
baseline_country = "US" #this was the highest occuring country in dataset

data = df["country"]
y_pred = [baseline_country] * len(data)

accuracy = (data == y_pred).mean()

print("Baseline country:", baseline_country)
print("Baseline accuracy:", accuracy)
print("Number of images:", len(data))
print("Number of correct guesses:", len(data) * accuracy)
print("Number of countries:", data.nunique())

Baseline country: US
Baseline accuracy: 0.11362446578654306
Number of images: 210122
Number of correct guesses: 23875.0
Number of countries: 219
